In [1]:
import torch
import torch.nn as nn
from transformers import AutoTokenizer, AutoModel, AutoModelForCausalLM


c:\Users\minhp\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [9]:
#Loading feature vectors
vision_features = torch.load('../Encoder/features.pt')
labels = torch.load('../Encoder/labels.pt')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
vision_features = vision_features.to(device)

In [11]:
#Text tokenizer and Encoder
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("microsoft/BiomedNLP-PubMedBERT-base-uncased-abstract")
text_encoder = AutoModel.from_pretrained("microsoft/BiomedNLP-PubMedBERT-base-uncased-abstract").to(device)

#Freeze text encoder parameters
for param in text_encoder.parameters():
    param.requires_grad = False

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 7506.87it/s]
BertModel LOAD REPORT from: microsoft/BiomedNLP-PubMedBERT-base-uncased-abstract
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [13]:
#LLM report generation

llm = AutoModelForCausalLM.from_pretrained("gpt2").to(device)

for p in llm.parameters():
    p.requires_grad = False

c:\Users\minhp\AppData\Local\Programs\Python\Python312\Lib\site-packages\huggingface_hub\file_download.py:129: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\minhp\.cache\huggingface\hub\models--gpt2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100%|██████████| 148/148 [00:00<00:00, 5426.29it/s]


In [14]:
#Diemensions
vision_dim = vision_features.shape[1]
text_dim = text_encoder.config.hidden_size
hidden_dim = 512
llm_dim = llm.config.n_embd

In [15]:
#Projection layer
image_projection = nn.Linear(vision_dim, hidden_dim).to(device)
text_projection = nn.Linear(text_dim, hidden_dim).to(device)
llm_projection = nn.Linear(llm_dim, hidden_dim).to(device)

In [ ]:
#Text encoder
def encode_text(text_list):
    inputs = tokenizer(
        text_list,
        padding=True,
        truncation=True,
        return_tensors="pt"
    ).to(device)

    with torch.no_grad():
        outputs = text_encoder(**inputs)
    
    #CLS token representation, summary of sentence
    return outputs.last_hidden_state[:, 0, :]

In [20]:
#Fusion module
def fuse(vision_emb, text_emb):
    return vision_emb + text_emb

In [ ]:
#Full forward pass

def forward(vision_feats, text_inputs):

    #Text
    text_emb = encode_text(text_inputs)
    text_emb = text_projection(text_emb)

    #Vision
    vision_emb = image_projection(vision_feats)

    #Fusion
    fused_emb = fuse(vision_emb, text_emb)

    #Project to LLM space
    llm_input = llm_projection(fused_emb)

    #gpt expects sequence
    llm_input = llm_input.unsqueeze(1)

    #Generate report
    outputs = llm(inputs_embeds=llm_input)

    return outputs.logits